# Tech Challenge - Analise Brazilian E-Commerce

## Relatorio Executivo para Investidores e Acionistas

**Base de dados:** Brazilian E-Commerce Public Dataset by Olist (~100 mil pedidos)  
**Periodo:** 2016 - 2018  
**Objetivo:** Transformar dados transacionais em insights estrategicos sobre desempenho comercial, eficiencia logistica e satisfacao do cliente.

---

### Trilhas Analiticas
1. Crescimento e Receita
2. Logistica e SLA
3. Comportamento e Pagamentos
4. Satisfacao do Cliente
5. Oportunidades e Recomendacoes
6. Text Mining de Reviews
7. Previsao com Sazonalidade e Cenarios
8. Analise de Coorte (Retencao)
9. Impacto Financeiro das Recomendacoes

---
## 0. Setup e Imports

In [1]:
import os
import re
import warnings
from collections import Counter
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# Configuracao visual profissional
sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
           '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.family': 'sans-serif',
})

%matplotlib inline

# Caminho dos dados (ajuste se necessario)
DATA_DIR = os.path.join('..', 'data')
print('Setup concluido!')

Setup concluido!


---
## 1. Carregamento dos Dados

O dataset e composto por **9 tabelas interconectadas** que cobrem todo o ecossistema do e-commerce brasileiro: clientes, pedidos, itens, produtos, vendedores, pagamentos, avaliacoes e geolocalizacao.

In [ ]:
# Carregar todos os CSVs
files = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

dfs = {}
for name, filename in files.items():
    path = os.path.join(DATA_DIR, filename)
    dfs[name] = pd.read_csv(path)
    print(f'  {name}: {dfs[name].shape[0]:,} linhas x {dfs[name].shape[1]} colunas')

print(f'\nTotal de tabelas carregadas: {len(dfs)}')

### Exploracao inicial dos dados

In [ ]:
# Visualizar as primeiras linhas da tabela principal de pedidos
print('=== ORDERS (Pedidos) ===')
display(dfs['orders'].head())
print(f'\nValores nulos:')
display(dfs['orders'].isnull().sum())

In [ ]:
# Explorar tabela de itens dos pedidos
print('=== ORDER ITEMS (Itens dos Pedidos) ===')
display(dfs['order_items'].head())
print(f'\nEstatisticas de preco e frete:')
display(dfs['order_items'][['price', 'freight_value']].describe())

In [ ]:
# Explorar avaliacoes
print('=== REVIEWS (Avaliacoes) ===')
display(dfs['reviews'].head())
print(f'\nDistribuicao de scores:')
display(dfs['reviews']['review_score'].value_counts().sort_index())

---
## 2. Limpeza e Preparacao dos Dados

Etapas:
- Conversao de colunas de data
- Calculo de lead times (aprovacao, postagem, transporte, total)
- Identificacao de atrasos (entrega real vs estimada)
- Traducao de categorias de produtos
- Merge entre tabelas para criar o DataFrame principal

In [ ]:
# Preparar tabela de pedidos com datas e lead times
orders = dfs['orders'].copy()
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

orders['order_year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
orders['order_weekday'] = orders['order_purchase_timestamp'].dt.day_name()

# Lead times
orders['lead_time_approval'] = (
    orders['order_approved_at'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 3600  # horas
orders['lead_time_carrier'] = (
    orders['order_delivered_carrier_date'] - orders['order_approved_at']
).dt.total_seconds() / 86400  # dias
orders['lead_time_delivery'] = (
    orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']
).dt.total_seconds() / 86400  # dias
orders['lead_time_total'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 86400  # dias
orders['delivery_vs_estimate'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.total_seconds() / 86400  # dias (positivo = atrasado)
orders['is_late'] = orders['delivery_vs_estimate'] > 0

print('Lead times calculados!')
print(f'Mediana lead time total: {orders["lead_time_total"].median():.1f} dias')
print(f'Taxa de atraso: {orders["is_late"].mean() * 100:.1f}%')

In [ ]:
# Traduzir categorias de produtos
products = dfs['products'].copy()
cat_trans = dfs['category_translation'].copy()
products = products.merge(cat_trans, on='product_category_name', how='left')
products['category'] = products['product_category_name_english'].fillna(
    products['product_category_name']
)
print(f'Categorias traduzidas: {products["category"].nunique()} categorias unicas')

In [ ]:
# Merge entre tabelas para criar DataFrame principal
order_items = dfs['order_items'].copy()
payments = dfs['payments'].copy()
reviews = dfs['reviews'].copy()
customers = dfs['customers'].copy()
sellers = dfs['sellers'].copy()

# Reviews: pegar apenas a mais recente por pedido
reviews = reviews.sort_values('review_creation_date').drop_duplicates(
    subset=['order_id'], keep='last'
)

# Merge items + products
items_products = order_items.merge(
    products[['product_id', 'category', 'product_category_name',
              'product_weight_g', 'product_length_cm',
              'product_height_cm', 'product_width_cm']],
    on='product_id', how='left',
)

# Merge orders + customers
orders_customers = orders.merge(
    customers[['customer_id', 'customer_unique_id',
               'customer_city', 'customer_state']],
    on='customer_id', how='left',
)

# DataFrame principal (nivel item)
main_df = items_products.merge(orders_customers, on='order_id', how='left')
main_df = main_df.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    on='seller_id', how='left',
)
main_df = main_df.merge(
    reviews[['order_id', 'review_score', 'review_comment_message']],
    on='order_id', how='left',
)

# Pagamentos agregados por pedido
payments_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    n_installments=('payment_installments', 'max'),
    payment_types=('payment_type', lambda x: ', '.join(x.unique())),
    primary_payment=('payment_type', 'first'),
).reset_index()

main_df = main_df.merge(payments_agg, on='order_id', how='left')

# Total por pedido (para analises a nivel pedido)
order_totals = order_items.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()
order_totals['total_order_value'] = order_totals['total_price'] + order_totals['total_freight']

orders_full = orders_customers.merge(order_totals, on='order_id', how='left')
orders_full = orders_full.merge(
    reviews[['order_id', 'review_score', 'review_comment_message']],
    on='order_id', how='left',
)
orders_full = orders_full.merge(payments_agg, on='order_id', how='left')

print(f'DataFrame principal (nivel item): {main_df.shape[0]:,} linhas x {main_df.shape[1]} colunas')
print(f'DataFrame pedidos completo: {orders_full.shape[0]:,} linhas x {orders_full.shape[1]} colunas')

### Funcao auxiliar

In [ ]:
def fmt_brl(value):
    """Formata valor em BRL."""
    return f'R$ {value:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

---
## 3. Trilha 1 - Crescimento e Receita

Analise da evolucao mensal de pedidos, receita, ticket medio, categorias e estados mais relevantes.

In [ ]:
# Filtrar pedidos validos
delivered = orders_full[
    orders_full['order_status'].isin(['delivered', 'shipped', 'invoiced', 'processing'])
].copy()
delivered = delivered.dropna(subset=['order_purchase_timestamp'])
delivered['year_month'] = delivered['order_purchase_timestamp'].dt.to_period('M')

monthly = delivered.groupby('year_month').agg(
    pedidos=('order_id', 'nunique'),
    receita=('total_order_value', 'sum'),
    ticket_medio=('total_order_value', 'mean'),
).reset_index()
monthly['year_month_str'] = monthly['year_month'].astype(str)

# KPIs principais
total_receita = delivered['total_order_value'].sum()
total_pedidos = delivered['order_id'].nunique()
ticket_medio_geral = delivered['total_order_value'].mean()

print(f'Receita Total: {fmt_brl(total_receita)}')
print(f'Total de Pedidos: {total_pedidos:,}')
print(f'Ticket Medio: {fmt_brl(ticket_medio_geral)}')
print(f'Clientes Unicos: {delivered["customer_unique_id"].nunique():,}')

In [ ]:
# Grafico 1.1: Evolucao mensal de pedidos e receita
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(monthly['year_month_str'], monthly['pedidos'], color=PALETTE[0], alpha=0.7, label='Pedidos')
ax1.set_xlabel('Mes')
ax1.set_ylabel('Numero de Pedidos', color=PALETTE[0])
ax1.tick_params(axis='y', labelcolor=PALETTE[0])
ax1.set_xticklabels(monthly['year_month_str'], rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(monthly['year_month_str'], monthly['receita'] / 1000, color=PALETTE[1],
         marker='o', linewidth=2.5, markersize=6, label='Receita (R$ mil)')
ax2.set_ylabel('Receita (R$ mil)', color=PALETTE[1])
ax2.tick_params(axis='y', labelcolor=PALETTE[1])

fig.suptitle('Evolucao Mensal de Pedidos e Receita', fontsize=16, fontweight='bold', y=1.02)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 1.2: Ticket medio mensal
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['year_month_str'], monthly['ticket_medio'], color=PALETTE[2],
        marker='s', linewidth=2.5, markersize=6)
ax.fill_between(range(len(monthly)), monthly['ticket_medio'], alpha=0.15, color=PALETTE[2])
ax.set_xticklabels(monthly['year_month_str'], rotation=45, ha='right')
ax.set_xlabel('Mes')
ax.set_ylabel('Ticket Medio (R$)')
ax.set_title('Evolucao do Ticket Medio Mensal', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('R$ %.0f'))
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 1.3: Top 10 categorias por receita
main_delivered = main_df[main_df['order_status'].isin(['delivered', 'shipped', 'invoiced', 'processing'])]
cat_revenue = main_delivered.groupby('category')['price'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cat_revenue.index[::-1], cat_revenue.values[::-1] / 1000, color=PALETTE[0], edgecolor='white')
ax.set_xlabel('Receita (R$ mil)')
ax.set_title('Top 10 Categorias por Receita', fontsize=14, fontweight='bold')
for bar in bars:
    width = bar.get_width()
    ax.text(width + 5, bar.get_y() + bar.get_height() / 2,
            f'R$ {width:,.0f}k', va='center', fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 1.4: Top 10 estados por receita
state_revenue = main_delivered.groupby('customer_state')['price'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors_state = sns.color_palette('Blues_r', len(state_revenue))
ax.bar(state_revenue.index, state_revenue.values / 1000, color=colors_state, edgecolor='white')
ax.set_xlabel('Estado')
ax.set_ylabel('Receita (R$ mil)')
ax.set_title('Top 10 Estados por Receita', fontsize=14, fontweight='bold')
for i, (idx, val) in enumerate(state_revenue.items()):
    ax.text(i, val / 1000 + 10, f'{val / state_revenue.sum() * 100:.1f}%',
            ha='center', fontsize=9, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 1.5: Top 10 sellers por receita
seller_rev = main_delivered.groupby('seller_id').agg(
    receita=('price', 'sum'),
    pedidos=('order_id', 'nunique'),
    cidade=('seller_city', 'first'),
    estado=('seller_state', 'first'),
).sort_values('receita', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
labels_sellers = [f'{row.cidade[:15]}-{row.estado}' for _, row in seller_rev.iterrows()]
ax.barh(labels_sellers[::-1], seller_rev['receita'].values[::-1] / 1000, color=PALETTE[1], edgecolor='white')
ax.set_xlabel('Receita (R$ mil)')
ax.set_title('Top 10 Sellers por Receita', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

---
## 4. Trilha 2 - Logistica e SLA

Analise dos tempos de entrega, etapas do processo logistico, impacto de atrasos na satisfacao e performance por estado.

In [ ]:
delivered_log = orders_full[orders_full['order_status'] == 'delivered'].dropna(subset=['lead_time_total']).copy()

median_lt = delivered_log['lead_time_total'].median()
pct_atraso = delivered_log['is_late'].mean() * 100

print(f'Mediana lead time total: {median_lt:.1f} dias')
print(f'Taxa de atraso: {pct_atraso:.1f}%')
print(f'Pedidos entregues analisados: {len(delivered_log):,}')

In [ ]:
# Grafico 2.1: Distribuicao do lead time total
fig, ax = plt.subplots(figsize=(12, 5))
lead_times = delivered_log['lead_time_total'].clip(0, 60)
ax.hist(lead_times, bins=60, color=PALETTE[0], alpha=0.8, edgecolor='white')
ax.axvline(median_lt, color=PALETTE[3], linestyle='--', linewidth=2,
           label=f'Mediana: {median_lt:.1f} dias')
ax.set_xlabel('Dias ate Entrega')
ax.set_ylabel('Numero de Pedidos')
ax.set_title('Distribuicao do Tempo Total de Entrega', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 2.2: Lead time por etapa (boxplot)
etapas = delivered_log[['lead_time_approval', 'lead_time_carrier', 'lead_time_delivery']].copy()
etapas.columns = ['Aprovacao (h)', 'Postagem (dias)', 'Transporte (dias)']
etapas['Aprovacao (h)'] = etapas['Aprovacao (h)'].clip(0, 72)
etapas['Postagem (dias)'] = etapas['Postagem (dias)'].clip(0, 15)
etapas['Transporte (dias)'] = etapas['Transporte (dias)'].clip(0, 40)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(etapas.columns):
    sns.boxplot(y=etapas[col], ax=axes[i], color=PALETTE[i], width=0.5)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
fig.suptitle('Tempo por Etapa do Processo', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 2.3: Atrasos vs review score
delivered_with_review = delivered_log.dropna(subset=['review_score', 'delivery_vs_estimate']).copy()
delivered_with_review['status_entrega'] = delivered_with_review['delivery_vs_estimate'].apply(
    lambda x: 'Atrasado' if x > 0 else 'No prazo/Adiantado'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score medio por status
score_by_status = delivered_with_review.groupby('status_entrega')['review_score'].mean()
colors_status = [PALETTE[3] if s == 'Atrasado' else PALETTE[2] for s in score_by_status.index]
axes[0].bar(score_by_status.index, score_by_status.values, color=colors_status, edgecolor='white')
axes[0].set_ylabel('Review Score Medio')
axes[0].set_title('Score Medio: Atrasados vs No Prazo', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 5.5)
for i, (idx, val) in enumerate(score_by_status.items()):
    axes[0].text(i, val + 0.1, f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

# Distribuicao de scores
for status, color in [('No prazo/Adiantado', PALETTE[2]), ('Atrasado', PALETTE[3])]:
    subset = delivered_with_review[delivered_with_review['status_entrega'] == status]
    score_dist = subset['review_score'].value_counts(normalize=True).sort_index()
    axes[1].bar(score_dist.index + (-0.2 if status == 'No prazo/Adiantado' else 0.2),
                score_dist.values * 100, width=0.35, color=color, alpha=0.8,
                label=status, edgecolor='white')
axes[1].set_xlabel('Review Score')
axes[1].set_ylabel('% dos Pedidos')
axes[1].set_title('Distribuicao de Scores por Status de Entrega', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].set_xticks([1, 2, 3, 4, 5])
fig.tight_layout()
plt.show()

print(f'Score medio - No prazo: {score_by_status.get("No prazo/Adiantado", 0):.2f}')
print(f'Score medio - Atrasado: {score_by_status.get("Atrasado", 0):.2f}')

In [ ]:
# Grafico 2.4: Taxa de atraso por estado
state_delivery = delivered_log.dropna(subset=['is_late']).groupby('customer_state').agg(
    total=('order_id', 'count'),
    atrasados=('is_late', 'sum'),
)
state_delivery['taxa_atraso'] = state_delivery['atrasados'] / state_delivery['total'] * 100
state_delivery = state_delivery[state_delivery['total'] >= 50].sort_values('taxa_atraso', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors_delay = [PALETTE[3] if v > 10 else PALETTE[0] for v in state_delivery['taxa_atraso']]
ax.bar(state_delivery.index, state_delivery['taxa_atraso'], color=colors_delay, edgecolor='white')
ax.set_xlabel('Estado')
ax.set_ylabel('Taxa de Atraso (%)')
ax.set_title('Taxa de Atraso na Entrega por Estado', fontsize=14, fontweight='bold')
ax.axhline(y=state_delivery['taxa_atraso'].mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'Media: {state_delivery["taxa_atraso"].mean():.1f}%')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 2.5: Lead time medio mensal
delivered_log['year_month'] = delivered_log['order_purchase_timestamp'].dt.to_period('M')
monthly_lt = delivered_log.groupby('year_month')['lead_time_total'].median().reset_index()
monthly_lt['year_month_str'] = monthly_lt['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_lt['year_month_str'], monthly_lt['lead_time_total'],
        color=PALETTE[0], marker='o', linewidth=2.5, markersize=6)
ax.fill_between(range(len(monthly_lt)), monthly_lt['lead_time_total'],
                alpha=0.15, color=PALETTE[0])
ax.set_xticklabels(monthly_lt['year_month_str'], rotation=45, ha='right')
ax.set_xlabel('Mes')
ax.set_ylabel('Mediana Lead Time (dias)')
ax.set_title('Evolucao Mensal do Tempo Mediano de Entrega', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

---
## 5. Trilha 3 - Comportamento e Pagamentos

Analise dos meios de pagamento, parcelamento, segmentacao RFM (Recencia, Frequencia, Monetizacao) e taxa de recompra.

In [ ]:
# Grafico 3.1: Meios de pagamento
pay_type = payments.groupby('payment_type').agg(
    total_valor=('payment_value', 'sum'),
    total_transacoes=('payment_type', 'count'),
).sort_values('total_transacoes', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels_pt = {
    'credit_card': 'Cartao Credito', 'boleto': 'Boleto',
    'voucher': 'Voucher', 'debit_card': 'Cartao Debito',
    'not_defined': 'Nao definido',
}
labels_pay = [labels_pt.get(x, x) for x in pay_type.index]
colors_pay = PALETTE[:len(pay_type)]

axes[0].pie(pay_type['total_transacoes'], labels=labels_pay, autopct='%1.1f%%',
            colors=colors_pay, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Transacoes por Meio de Pagamento', fontsize=12, fontweight='bold')

axes[1].pie(pay_type['total_valor'], labels=labels_pay, autopct='%1.1f%%',
            colors=colors_pay, startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Volume Financeiro por Meio de Pagamento', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 3.2: Distribuicao de parcelas no cartao
credit = payments[payments['payment_type'] == 'credit_card'].copy()
installments_dist = credit.groupby('payment_installments').agg(
    count=('payment_value', 'count'),
    avg_value=('payment_value', 'mean'),
).reset_index()
installments_dist = installments_dist[installments_dist['payment_installments'] <= 12]

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(installments_dist['payment_installments'], installments_dist['count'],
        color=PALETTE[0], alpha=0.7, label='Qtd Transacoes')
ax1.set_xlabel('Numero de Parcelas')
ax1.set_ylabel('Quantidade de Transacoes', color=PALETTE[0])
ax1.tick_params(axis='y', labelcolor=PALETTE[0])

ax2 = ax1.twinx()
ax2.plot(installments_dist['payment_installments'], installments_dist['avg_value'],
         color=PALETTE[1], marker='o', linewidth=2.5, label='Valor Medio (R$)')
ax2.set_ylabel('Valor Medio da Compra (R$)', color=PALETTE[1])
ax2.tick_params(axis='y', labelcolor=PALETTE[1])

fig.suptitle('Distribuicao de Parcelas no Cartao de Credito', fontsize=14, fontweight='bold', y=1.02)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.set_xticks(range(1, 13))
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 3.3: Analise RFM (Recencia, Frequencia, Monetizacao)
delivered_rfm = orders_full[orders_full['order_status'] == 'delivered'].dropna(
    subset=['order_purchase_timestamp', 'total_order_value']
).copy()
reference_date = delivered_rfm['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = delivered_rfm.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_order_value', 'sum'),
).reset_index()

rfm['R_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1])
rfm['F_score'] = pd.cut(rfm['frequency'], bins=[0, 1, 2, 3, 100], labels=[1, 2, 3, 4])
rfm['M_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4])
rfm['RFM_score'] = rfm[['R_score', 'F_score', 'M_score']].astype(int).sum(axis=1)

def rfm_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal'
    elif score >= 6: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['segment'] = rfm['RFM_score'].apply(rfm_segment)

segment_summary = rfm.groupby('segment').agg(
    clientes=('customer_unique_id', 'count'),
    receita_media=('monetary', 'mean'),
    recencia_media=('recency', 'mean'),
).sort_values('clientes', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
seg_colors = {'Champions': PALETTE[2], 'Loyal': PALETTE[0], 'Potential': PALETTE[1],
              'At Risk': PALETTE[4], 'Lost': PALETTE[3]}
colors_seg = [seg_colors.get(s, 'gray') for s in segment_summary.index]

axes[0].pie(segment_summary['clientes'], labels=segment_summary.index, autopct='%1.1f%%',
            colors=colors_seg, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Distribuicao de Clientes por Segmento RFM', fontsize=12, fontweight='bold')

axes[1].barh(segment_summary.index[::-1], segment_summary['receita_media'].values[::-1],
             color=[seg_colors.get(s, 'gray') for s in segment_summary.index[::-1]], edgecolor='white')
axes[1].set_xlabel('Receita Media por Cliente (R$)')
axes[1].set_title('Receita Media por Segmento RFM', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

display(segment_summary)

In [ ]:
# Grafico 3.4: Taxa de recompra
customer_orders = delivered_rfm.groupby('customer_unique_id')['order_id'].nunique()
recompra = (customer_orders > 1).sum()
total_customers = len(customer_orders)
taxa_recompra = recompra / total_customers * 100

fig, ax = plt.subplots(figsize=(8, 6))
sizes = [total_customers - recompra, recompra]
labels_rec = [f'Compra unica\n({total_customers - recompra:,})', f'Recompra\n({recompra:,})']
colors_rec = [PALETTE[0], PALETTE[2]]
ax.pie(sizes, labels=labels_rec, autopct='%1.1f%%',
       colors=colors_rec, startangle=90, textprops={'fontsize': 11})
ax.set_title(f'Taxa de Recompra: {taxa_recompra:.1f}%', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print(f'Clientes unicos: {total_customers:,}')
print(f'Clientes com recompra: {recompra:,}')
print(f'Taxa de recompra: {taxa_recompra:.1f}%')

---
## 6. Trilha 4 - Satisfacao do Cliente

Distribuicao de scores de avaliacao, drivers de satisfacao (entrega e preco) e evolucao do NPS (Net Promoter Score).

In [ ]:
delivered_sat = orders_full[orders_full['order_status'] == 'delivered'].dropna(subset=['review_score']).copy()

score_medio = delivered_sat['review_score'].mean()
pct_5 = (delivered_sat['review_score'] == 5).mean() * 100
pct_1 = (delivered_sat['review_score'] == 1).mean() * 100
nps_geral = ((delivered_sat['review_score'] >= 4).sum() - (delivered_sat['review_score'] <= 2).sum()) / len(delivered_sat) * 100

print(f'Score medio: {score_medio:.2f}/5.0')
print(f'5 estrelas: {pct_5:.1f}%')
print(f'1 estrela: {pct_1:.1f}%')
print(f'NPS geral: {nps_geral:.1f}')

In [ ]:
# Grafico 4.1: Distribuicao de review scores
score_dist = delivered_sat['review_score'].value_counts().sort_index()
total = score_dist.sum()

fig, ax = plt.subplots(figsize=(10, 6))
colors_score = [PALETTE[3], PALETTE[3], PALETTE[1], PALETTE[2], PALETTE[2]]
bars = ax.bar(score_dist.index, score_dist.values, color=colors_score, edgecolor='white', width=0.7)
ax.set_xlabel('Review Score')
ax.set_ylabel('Numero de Pedidos')
ax.set_title('Distribuicao das Avaliacoes dos Clientes', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
for bar, val in zip(bars, score_dist.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{val:,}\n({val / total * 100:.1f}%)', ha='center', fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 4.2: Score medio por categoria (top 15)
main_del = main_df[main_df['order_status'] == 'delivered'].dropna(subset=['review_score', 'category'])
cat_score = main_del.groupby('category').agg(
    score_medio=('review_score', 'mean'),
    count=('order_id', 'count'),
)
cat_score = cat_score[cat_score['count'] >= 100].sort_values('score_medio', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors_cat = [PALETTE[2] if s >= 4 else PALETTE[1] if s >= 3.5 else PALETTE[3] for s in cat_score['score_medio']]
ax.barh(cat_score.index, cat_score['score_medio'], color=colors_cat, edgecolor='white')
ax.set_xlabel('Score Medio')
ax.set_xlim(2.5, 5)
ax.set_title('Score Medio por Categoria (min. 100 pedidos)', fontsize=14, fontweight='bold')
ax.axvline(x=4.0, color='gray', linestyle='--', alpha=0.5)
for i, (idx, row) in enumerate(cat_score.iterrows()):
    ax.text(row['score_medio'] + 0.03, i, f'{row["score_medio"]:.2f}', va='center', fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 4.3: Drivers de satisfacao
drivers = delivered_sat.dropna(subset=['lead_time_total', 'total_order_value']).copy()
drivers['faixa_lead_time'] = pd.cut(drivers['lead_time_total'],
                                     bins=[0, 5, 10, 15, 20, 30, 100],
                                     labels=['0-5d', '5-10d', '10-15d', '15-20d', '20-30d', '30d+'])
drivers['faixa_preco'] = pd.cut(drivers['total_order_value'],
                                 bins=[0, 50, 100, 200, 500, 10000],
                                 labels=['<R$50', 'R$50-100', 'R$100-200', 'R$200-500', 'R$500+'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lt_score = drivers.groupby('faixa_lead_time')['review_score'].mean()
axes[0].bar(lt_score.index.astype(str), lt_score.values, color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Tempo de Entrega')
axes[0].set_ylabel('Score Medio')
axes[0].set_title('Score por Tempo de Entrega', fontsize=12, fontweight='bold')
axes[0].set_ylim(2, 5)

price_score = drivers.groupby('faixa_preco')['review_score'].mean()
axes[1].bar(price_score.index.astype(str), price_score.values, color=PALETTE[1], edgecolor='white')
axes[1].set_xlabel('Faixa de Preco')
axes[1].set_ylabel('Score Medio')
axes[1].set_title('Score por Faixa de Preco', fontsize=12, fontweight='bold')
axes[1].set_ylim(2, 5)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 4.4: Evolucao mensal do NPS
delivered_sat['year_month'] = delivered_sat['order_purchase_timestamp'].dt.to_period('M')
monthly_nps = delivered_sat.groupby('year_month').apply(
    lambda x: ((x['review_score'] >= 4).sum() - (x['review_score'] <= 2).sum()) / len(x) * 100
).reset_index(name='nps')
monthly_nps['year_month_str'] = monthly_nps['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
colors_nps = [PALETTE[2] if n >= 50 else PALETTE[1] if n >= 30 else PALETTE[3] for n in monthly_nps['nps']]
ax.bar(monthly_nps['year_month_str'], monthly_nps['nps'], color=colors_nps, edgecolor='white')
ax.set_xticklabels(monthly_nps['year_month_str'], rotation=45, ha='right')
ax.set_xlabel('Mes')
ax.set_ylabel('NPS (%)')
ax.set_title('Evolucao Mensal do NPS (Net Promoter Score)', fontsize=14, fontweight='bold')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Zona de excelencia (50+)')
ax.legend()
fig.tight_layout()
plt.show()

---
## 7. Trilha 5 - Oportunidades e Recomendacoes

Analise de frete por categoria, performance de sellers, oportunidades de cross-sell e heatmap de demanda.

In [ ]:
main_del_opp = main_df[main_df['order_status'] == 'delivered'].copy()

# Grafico 5.1: Frete como % do preco por categoria
cat_freight = main_del_opp.groupby('category').agg(
    preco_medio=('price', 'mean'),
    frete_medio=('freight_value', 'mean'),
    count=('order_id', 'count'),
)
cat_freight['frete_pct'] = cat_freight['frete_medio'] / cat_freight['preco_medio'] * 100
cat_freight = cat_freight[cat_freight['count'] >= 100].sort_values('frete_pct', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors_freight = [PALETTE[3] if f > 30 else PALETTE[1] if f > 20 else PALETTE[2]
                  for f in cat_freight['frete_pct']]
ax.barh(cat_freight.index[::-1], cat_freight['frete_pct'].values[::-1],
        color=colors_freight[::-1], edgecolor='white')
ax.set_xlabel('Frete como % do Preco do Produto')
ax.set_title('Categorias com Maior Peso de Frete', fontsize=14, fontweight='bold')
ax.axvline(x=20, color='gray', linestyle='--', alpha=0.5, label='Limite 20%')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 5.2: Sellers - Score vs Lead Time (matriz de performance)
seller_perf = main_del_opp.dropna(subset=['review_score', 'lead_time_total']).groupby('seller_id').agg(
    score_medio=('review_score', 'mean'),
    lead_time_medio=('lead_time_total', 'mean'),
    receita=('price', 'sum'),
    pedidos=('order_id', 'nunique'),
).reset_index()
seller_perf = seller_perf[seller_perf['pedidos'] >= 10]

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    seller_perf['lead_time_medio'], seller_perf['score_medio'],
    s=seller_perf['receita'] / 500, alpha=0.5, c=seller_perf['score_medio'],
    cmap='RdYlGn', edgecolors='white', linewidth=0.5,
)
ax.set_xlabel('Lead Time Medio (dias)')
ax.set_ylabel('Score Medio')
ax.set_title('Performance dos Sellers: Score vs Lead Time\n(tamanho = receita)',
             fontsize=14, fontweight='bold')
ax.axhline(y=4, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=15, color='gray', linestyle='--', alpha=0.5)
ax.text(5, 4.8, 'IDEAL', fontsize=12, color=PALETTE[2], fontweight='bold', alpha=0.7)
ax.text(25, 2, 'CRITICO', fontsize=12, color=PALETTE[3], fontweight='bold', alpha=0.7)
plt.colorbar(scatter, label='Score Medio', ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 5.3: Cross-sell - categorias compradas juntas
multi_item_orders = main_del_opp.groupby('order_id')['category'].apply(list)
multi_item_orders = multi_item_orders[multi_item_orders.apply(len) > 1]

pair_counts = Counter()
for cats in multi_item_orders:
    unique_cats = list(set(cats))
    if len(unique_cats) > 1:
        for pair in combinations(sorted(unique_cats), 2):
            pair_counts[pair] += 1

top_pairs = pair_counts.most_common(15)

fig, ax = plt.subplots(figsize=(14, 7))
pair_labels = [f'{p[0][:20]} +\n{p[1][:20]}' for p, c in top_pairs]
pair_values = [c for p, c in top_pairs]
ax.barh(pair_labels[::-1], pair_values[::-1], color=PALETTE[0], edgecolor='white')
ax.set_xlabel('Numero de Pedidos')
ax.set_title('Top 15 Combinacoes de Categorias (Cross-Sell)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 5.4: Heatmap de demanda por dia da semana e hora
orders_heatmap = orders.dropna(subset=['order_purchase_timestamp']).copy()
orders_heatmap['hour'] = orders_heatmap['order_purchase_timestamp'].dt.hour
orders_heatmap['weekday'] = orders_heatmap['order_purchase_timestamp'].dt.day_name()

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_labels = ['Segunda', 'Terca', 'Quarta', 'Quinta', 'Sexta', 'Sabado', 'Domingo']

heatmap_data = orders_heatmap.groupby(['weekday', 'hour']).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(weekday_order)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Numero de Pedidos'})
ax.set_yticklabels(weekday_labels, rotation=0)
ax.set_xlabel('Hora do Dia')
ax.set_ylabel('Dia da Semana')
ax.set_title('Heatmap de Demanda: Dia da Semana vs Hora', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 5.5: Projecao de tendencia de pedidos
delivered_proj = orders_full[
    orders_full['order_status'].isin(['delivered', 'shipped', 'invoiced', 'processing'])
].dropna(subset=['order_purchase_timestamp']).copy()
delivered_proj['year_month'] = delivered_proj['order_purchase_timestamp'].dt.to_period('M')

monthly_proj = delivered_proj.groupby('year_month').agg(
    pedidos=('order_id', 'nunique'),
).reset_index()
monthly_proj['month_num'] = range(len(monthly_proj))
monthly_proj['year_month_str'] = monthly_proj['year_month'].astype(str)

# Remover meses incompletos
monthly_proj_clean = monthly_proj.iloc[1:-1]
x_proj = monthly_proj_clean['month_num'].values
y_proj = monthly_proj_clean['pedidos'].values
coeffs_proj = np.polyfit(x_proj, y_proj, 1)
trend_proj = np.poly1d(coeffs_proj)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(monthly_proj['year_month_str'], monthly_proj['pedidos'], color=PALETTE[0], alpha=0.7, label='Realizado')
all_x_proj = range(len(monthly_proj) + 3)
ax.plot(range(len(monthly_proj)), trend_proj(monthly_proj['month_num']),
        color=PALETTE[3], linestyle='--', linewidth=2, label='Tendencia')
ax.set_xticklabels(monthly_proj['year_month_str'], rotation=45, ha='right')
ax.set_xlabel('Mes')
ax.set_ylabel('Numero de Pedidos')
ax.set_title('Tendencia de Crescimento de Pedidos', fontsize=14, fontweight='bold')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Crescimento medio mensal: +{coeffs_proj[0]:.0f} pedidos/mes')

---
## 8. Text Mining de Reviews

Analise das palavras mais frequentes em avaliacoes positivas (nota 5) vs negativas (nota 1), e identificacao de temas recorrentes nas avaliacoes.

In [ ]:
reviews_tm = orders_full.dropna(subset=['review_comment_message', 'review_score']).copy()

# Stopwords em portugues
stopwords_pt = {
    'a', 'o', 'e', 'de', 'da', 'do', 'em', 'um', 'uma', 'que', 'para', 'com',
    'nao', 'por', 'mais', 'se', 'no', 'na', 'os', 'as', 'dos', 'das', 'ao',
    'mas', 'ou', 'foi', 'tem', 'ser', 'eu', 'me', 'meu', 'minha', 'muito',
    'ja', 'como', 'sua', 'seu', 'ele', 'ela', 'nos', 'esse', 'essa', 'isso',
    'este', 'esta', 'aqui', 'ali', 'la', 'bem', 'so', 'ate', 'pra', 'tb',
    'ainda', 'depois', 'antes', 'agora', 'quando', 'onde', 'qual', 'quem',
    'tudo', 'nada', 'cada', 'mesmo', 'outro', 'outra', 'outros', 'outras',
    'voce', 'vc', 'estou', 'esta', 'estao', 'vai', 'vou', 'pode', 'ter',
    'tinha', 'tive', 'fiz', 'fazer', 'feito', 'dia', 'dias', 'vez', 'vezes',
    'sim', 'tambem', 'entao', 'pois', 'porque', 'sobre', 'entre', 'sem',
    'pelo', 'pela', 'pelos', 'pelas', 'num', 'numa', 'nos', 'das', 'aos',
    'uns', 'umas', 'lhe', 'lhes', 'the', 'and', 'to', 'of', 'is', 'it',
    'in', 'for', 'on', 'with', 'not', 'are', 'was', 'were', 'be', 'have',
    'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should',
    'all', 'any', 'some', 'this', 'that', 'these', 'those', 'from', 'they',
    'you', 'we', 'our', 'your', 'their', 'its', 'my', 'his', 'her',
}

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zaaaaeeeiioooouuc\s]', ' ', text)
    words = text.split()
    return [w for w in words if len(w) > 2 and w not in stopwords_pt]

reviews_1 = reviews_tm[reviews_tm['review_score'] == 1]['review_comment_message']
reviews_5 = reviews_tm[reviews_tm['review_score'] == 5]['review_comment_message']

words_1 = Counter()
for text in reviews_1:
    words_1.update(tokenize(text))

words_5 = Counter()
for text in reviews_5:
    words_5.update(tokenize(text))

print(f'Reviews nota 1: {len(reviews_1):,}')
print(f'Reviews nota 5: {len(reviews_5):,}')

In [ ]:
# Grafico 6.1: Top 20 palavras em reviews nota 1 vs nota 5
top_neg = words_1.most_common(20)
top_pos = words_5.most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

words_neg, counts_neg = zip(*top_neg)
axes[0].barh(list(words_neg)[::-1], list(counts_neg)[::-1], color=PALETTE[3], edgecolor='white')
axes[0].set_title('Top 20 Palavras - Reviews Nota 1\n(Insatisfeitos)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frequencia')

words_pos, counts_pos = zip(*top_pos)
axes[1].barh(list(words_pos)[::-1], list(counts_pos)[::-1], color=PALETTE[2], edgecolor='white')
axes[1].set_title('Top 20 Palavras - Reviews Nota 5\n(Satisfeitos)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequencia')

fig.suptitle('Analise de Text Mining: O que Clientes Dizem?', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 6.2: Temas recorrentes nas avaliacoes
temas_negativos = {
    'atraso/entrega': ['entrega', 'chegou', 'atraso', 'atrasado', 'prazo', 'demora',
                       'demorou', 'demais', 'dias', 'esperando'],
    'produto/qualidade': ['produto', 'qualidade', 'veio', 'diferente', 'quebrado',
                          'danificado', 'defeito', 'ruim'],
    'comunicacao': ['vendedor', 'resposta', 'contato', 'informacao', 'rastreamento',
                    'rastreio', 'tracking'],
}
temas_positivos = {
    'entrega_rapida': ['rapido', 'rapida', 'antes', 'prazo', 'antecipado', 'cedo'],
    'qualidade': ['otimo', 'otima', 'excelente', 'perfeito', 'perfeita', 'qualidade',
                  'bom', 'boa', 'lindo', 'linda', 'bonito'],
    'recomendacao': ['recomendo', 'amei', 'adorei', 'satisfeito', 'satisfeita',
                     'parabens', 'obrigado', 'obrigada', 'gostei'],
}

def count_theme(texts, keywords):
    count = 0
    for text in texts:
        text_lower = str(text).lower()
        if any(kw in text_lower for kw in keywords):
            count += 1
    return count

reviews_1_list = reviews_1.tolist()
reviews_5_list = reviews_5.tolist()

tema_neg_counts = {tema: count_theme(reviews_1_list, kws) for tema, kws in temas_negativos.items()}
tema_pos_counts = {tema: count_theme(reviews_5_list, kws) for tema, kws in temas_positivos.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels_neg = {'atraso/entrega': 'Atraso/Entrega', 'produto/qualidade': 'Produto/Qualidade',
              'comunicacao': 'Comunicacao'}
labels_pos = {'entrega_rapida': 'Entrega Rapida', 'qualidade': 'Qualidade',
              'recomendacao': 'Recomendacao'}

neg_labels = [labels_neg[k] for k in tema_neg_counts]
neg_values = list(tema_neg_counts.values())
neg_pcts = [v / len(reviews_1_list) * 100 for v in neg_values]
axes[0].bar(neg_labels, neg_pcts, color=[PALETTE[3], PALETTE[1], PALETTE[4]], edgecolor='white')
axes[0].set_ylabel('% das Reviews Nota 1')
axes[0].set_title('Temas em Reviews Negativas', fontsize=12, fontweight='bold')
for i, pct in enumerate(neg_pcts):
    axes[0].text(i, pct + 1, f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')

pos_labels = [labels_pos[k] for k in tema_pos_counts]
pos_values = list(tema_pos_counts.values())
pos_pcts = [v / len(reviews_5_list) * 100 for v in pos_values]
axes[1].bar(pos_labels, pos_pcts, color=[PALETTE[2], PALETTE[0], PALETTE[4]], edgecolor='white')
axes[1].set_ylabel('% das Reviews Nota 5')
axes[1].set_title('Temas em Reviews Positivas', fontsize=12, fontweight='bold')
for i, pct in enumerate(pos_pcts):
    axes[1].text(i, pct + 1, f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')

fig.suptitle('Temas Recorrentes nas Avaliacoes', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

---
## 9. Previsao com Sazonalidade e Cenarios

Projecao de receita utilizando tendencia linear + fatores de sazonalidade mensal, com cenarios otimista (+15%) e pessimista (-15%).

In [ ]:
delivered_prev = orders_full[
    orders_full['order_status'].isin(['delivered', 'shipped', 'invoiced', 'processing'])
].dropna(subset=['order_purchase_timestamp']).copy()

delivered_prev['year_month'] = delivered_prev['order_purchase_timestamp'].dt.to_period('M')
monthly_prev = delivered_prev.groupby('year_month').agg(
    receita=('total_order_value', 'sum'),
    pedidos=('order_id', 'nunique'),
).reset_index()
monthly_prev['month_num'] = range(len(monthly_prev))
monthly_prev['year_month_str'] = monthly_prev['year_month'].astype(str)

# Remover meses incompletos
monthly_clean = monthly_prev.iloc[1:-1].copy()
monthly_clean['receita_mm3'] = monthly_clean['receita'].rolling(window=3, center=True).mean()

# Tendencia linear
x = monthly_clean['month_num'].values
y = monthly_clean['receita'].values
coeffs = np.polyfit(x, y, 1)
trend = np.poly1d(coeffs)

# Sazonalidade
monthly_clean['trend_value'] = trend(monthly_clean['month_num'])
monthly_clean['seasonal_ratio'] = monthly_clean['receita'] / monthly_clean['trend_value']
monthly_clean['month_of_year'] = monthly_clean['year_month'].apply(lambda x: x.month)
seasonal_factors = monthly_clean.groupby('month_of_year')['seasonal_ratio'].mean()

# Projecao 6 meses
last_month_num = monthly_prev.iloc[-1]['month_num']
last_period = monthly_prev.iloc[-1]['year_month']

future_data = []
for i in range(1, 7):
    month_num = last_month_num + i
    period = last_period + i
    month_of_year = period.month
    base_value = trend(month_num)
    seasonal_factor = seasonal_factors.get(month_of_year, 1.0)
    projected = base_value * seasonal_factor
    future_data.append({
        'year_month_str': str(period),
        'month_num': month_num,
        'receita_projetada': projected,
        'seasonal_factor': seasonal_factor,
    })

future_df = pd.DataFrame(future_data)
future_df['cenario_otimista'] = future_df['receita_projetada'] * 1.15
future_df['cenario_pessimista'] = future_df['receita_projetada'] * 0.85

print(f'Receita projetada (6 meses): {fmt_brl(future_df["receita_projetada"].sum())}')
print(f'Cenario otimista: {fmt_brl(future_df["cenario_otimista"].sum())}')
print(f'Cenario pessimista: {fmt_brl(future_df["cenario_pessimista"].sum())}')

In [ ]:
# Grafico 7.1: Previsao com cenarios
fig, ax = plt.subplots(figsize=(16, 7))

ax.bar(range(len(monthly_prev)), monthly_prev['receita'] / 1000, color=PALETTE[0], alpha=0.7, label='Realizado')

# Media movel
ax.plot([i + 1 for i in range(len(monthly_clean))],
        monthly_clean['receita_mm3'].values / 1000,
        color=PALETTE[4], linewidth=2.5, linestyle='-', label='Media Movel 3M')

# Projecao
future_x = [len(monthly_prev) + i for i in range(len(future_df))]
ax.bar(future_x, future_df['receita_projetada'] / 1000, color=PALETTE[1], alpha=0.6,
       label='Projecao (com sazonalidade)')

# Cenarios
ax.fill_between(future_x,
                future_df['cenario_pessimista'] / 1000,
                future_df['cenario_otimista'] / 1000,
                alpha=0.2, color=PALETTE[1], label='Intervalo +/-15%')

# Tendencia
all_x = list(range(len(monthly_prev) + len(future_df)))
ax.plot(all_x, trend(all_x) / 1000, color=PALETTE[3], linestyle='--', linewidth=1.5,
        label='Tendencia Linear')

all_labels = list(monthly_prev['year_month_str']) + list(future_df['year_month_str'])
ax.set_xticks(range(0, len(all_labels), 2))
ax.set_xticklabels([all_labels[i] for i in range(0, len(all_labels), 2)], rotation=45, ha='right')
ax.set_xlabel('Mes')
ax.set_ylabel('Receita (R$ mil)')
ax.set_title('Previsao de Receita com Sazonalidade e Cenarios', fontsize=15, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.axvline(x=len(monthly_prev) - 0.5, color='gray', linestyle=':', alpha=0.7)
ax.text(len(monthly_prev) + 1, ax.get_ylim()[1] * 0.95, 'PROJECAO', fontsize=11,
        color='gray', fontweight='bold', ha='center')
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 7.2: Fatores de sazonalidade
fig, ax = plt.subplots(figsize=(10, 5))
month_names = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
               'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
available_months = sorted(seasonal_factors.index)
labels_saz = [month_names[m - 1] for m in available_months]
values_saz = [seasonal_factors[m] for m in available_months]
colors_season = [PALETTE[2] if v >= 1 else PALETTE[3] for v in values_saz]

ax.bar(labels_saz, values_saz, color=colors_season, edgecolor='white')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='Neutro (1.0)')
ax.set_xlabel('Mes')
ax.set_ylabel('Fator de Sazonalidade')
ax.set_title('Fatores de Sazonalidade Mensal', fontsize=14, fontweight='bold')
ax.legend()
for i, v in enumerate(values_saz):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
fig.tight_layout()
plt.show()

---
## 10. Analise de Coorte (Retencao)

Acompanhamento da retencao de clientes ao longo do tempo, agrupados pelo mes da primeira compra. Fundamental para entender a fidelizacao e o ciclo de vida do cliente.

In [ ]:
delivered_coorte = orders_full[orders_full['order_status'] == 'delivered'].dropna(
    subset=['order_purchase_timestamp', 'customer_unique_id']
).copy()

# Coorte = mes da primeira compra
first_purchase = delivered_coorte.groupby('customer_unique_id')['order_purchase_timestamp'].min().reset_index()
first_purchase.columns = ['customer_unique_id', 'first_purchase']
first_purchase['cohort'] = first_purchase['first_purchase'].dt.to_period('M')

delivered_coorte = delivered_coorte.merge(first_purchase, on='customer_unique_id', how='left')
delivered_coorte['order_period'] = delivered_coorte['order_purchase_timestamp'].dt.to_period('M')
delivered_coorte['cohort_month'] = (
    (delivered_coorte['order_period'] - delivered_coorte['cohort']).apply(lambda x: x.n if hasattr(x, 'n') else 0)
)

cohort_data = delivered_coorte.groupby(['cohort', 'cohort_month'])['customer_unique_id'].nunique().reset_index()
cohort_data.columns = ['cohort', 'cohort_month', 'customers']

cohort_pivot = cohort_data.pivot_table(
    index='cohort', columns='cohort_month', values='customers', fill_value=0
)

cohort_sizes = cohort_pivot[0]
retention = cohort_pivot.divide(cohort_sizes, axis=0) * 100

valid_cohorts = cohort_sizes[cohort_sizes >= 100].index
retention_filtered = retention.loc[valid_cohorts]
max_months = min(12, retention_filtered.columns.max() + 1)
retention_filtered = retention_filtered.iloc[:, :max_months]
retention_filtered = retention_filtered.loc[retention_filtered.notna().sum(axis=1) >= 3]

avg_retention = retention_filtered.mean()
print(f'Retencao media mes 1: {avg_retention.get(1, 0):.2f}%')
print(f'Retencao media mes 3: {avg_retention.get(3, 0):.2f}%')
print(f'Retencao media mes 6: {avg_retention.get(6, 0):.2f}%')

In [ ]:
# Grafico 8.1: Heatmap de retencao por coorte
fig, ax = plt.subplots(figsize=(16, 10))
labels_ret = retention_filtered.round(1)
sns.heatmap(retention_filtered, annot=labels_ret, fmt='.1f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Taxa de Retencao (%)'},
            vmin=0, vmax=10)
ax.set_xlabel('Meses desde Primeira Compra')
ax.set_ylabel('Coorte (Mes da Primeira Compra)')
ax.set_title('Analise de Retencao por Coorte', fontsize=15, fontweight='bold')
ax.set_yticklabels([str(c) for c in retention_filtered.index], rotation=0)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 8.2: Curva de retencao media
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(avg_retention.index, avg_retention.values, color=PALETTE[0],
        marker='o', linewidth=2.5, markersize=8)
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.15, color=PALETTE[0])
ax.set_xlabel('Meses desde Primeira Compra')
ax.set_ylabel('Taxa de Retencao Media (%)')
ax.set_title('Curva de Retencao Media (todas as coortes)', fontsize=14, fontweight='bold')
ax.set_xticks(avg_retention.index)
for i, v in enumerate(avg_retention.values):
    if i <= 6 or i == len(avg_retention) - 1:
        ax.text(avg_retention.index[i], v + 0.2, f'{v:.1f}%', ha='center', fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 8.3: Tamanho das coortes
cohort_sizes_filtered = cohort_sizes.loc[valid_cohorts]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(cohort_sizes_filtered)),
       cohort_sizes_filtered.values,
       color=PALETTE[0], edgecolor='white')
ax.set_xticks(range(len(cohort_sizes_filtered)))
ax.set_xticklabels([str(c) for c in cohort_sizes_filtered.index], rotation=45, ha='right')
ax.set_xlabel('Coorte (Mes da Primeira Compra)')
ax.set_ylabel('Novos Clientes')
ax.set_title('Aquisicao de Novos Clientes por Coorte', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

---
## 11. Impacto Financeiro das Recomendacoes

Estimativas do impacto financeiro de cada iniciativa estrategica recomendada, com base nos dados reais do dataset.

In [ ]:
delivered_fin = orders_full[orders_full['order_status'] == 'delivered'].dropna(
    subset=['total_order_value']).copy()

total_receita_fin = delivered_fin['total_order_value'].sum()
total_clientes_fin = delivered_fin['customer_unique_id'].nunique()
ticket_medio_fin = delivered_fin['total_order_value'].mean()

# Cenario 1: Dobrar taxa de recompra (3% -> 6%)
novos_recompra = int(total_clientes_fin * 0.03)
receita_incremental_recompra = novos_recompra * ticket_medio_fin

# Cenario 2: Cross-sell (+10% no ticket medio)
receita_incremental_crosssell = total_receita_fin * 0.10

# Cenario 3: Otimizacao de frete
cat_freight_fin = main_del_opp.groupby('category').agg(
    preco_medio=('price', 'mean'),
    frete_medio=('freight_value', 'mean'),
    count=('order_id', 'count'),
    receita=('price', 'sum'),
)
cat_freight_fin['frete_pct'] = cat_freight_fin['frete_medio'] / cat_freight_fin['preco_medio'] * 100
receita_incremental_frete = cat_freight_fin[cat_freight_fin['frete_pct'] > 30]['receita'].sum() * 0.05

receita_total_incremental = receita_incremental_recompra + receita_incremental_crosssell + receita_incremental_frete
aumento_pct = receita_total_incremental / total_receita_fin * 100

print(f'Impacto estimado por iniciativa:')
print(f'  Programa de recompra: {fmt_brl(receita_incremental_recompra)}')
print(f'  Cross-sell inteligente: {fmt_brl(receita_incremental_crosssell)}')
print(f'  Otimizacao de frete: {fmt_brl(receita_incremental_frete)}')
print(f'  TOTAL: {fmt_brl(receita_total_incremental)} (+{aumento_pct:.1f}%)')

In [ ]:
# Grafico 9.1: Impacto financeiro por iniciativa
initiatives = {
    'Programa de\nRecompra\n(3% -> 6%)': receita_incremental_recompra,
    'Cross-Sell\nInteligente\n(+10% ticket)': receita_incremental_crosssell,
    'Otimizacao\nde Frete\n(cats criticas)': receita_incremental_frete,
}

fig, ax = plt.subplots(figsize=(12, 6))
names = list(initiatives.keys())
values_init = [v / 1_000_000 for v in initiatives.values()]
colors_init = [PALETTE[2], PALETTE[0], PALETTE[1]]

bars = ax.bar(names, values_init, color=colors_init, edgecolor='white', width=0.6)
ax.set_ylabel('Receita Incremental Estimada (R$ milhoes)')
ax.set_title('Impacto Financeiro Estimado por Iniciativa Estrategica', fontsize=14, fontweight='bold')
for bar, val in zip(bars, values_init):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'R$ {val:.2f}M', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(values_init) * 1.3)
fig.tight_layout()
plt.show()

In [ ]:
# Grafico 9.2: Receita projetada com iniciativas combinadas
receita_projetada_com = total_receita_fin + receita_total_incremental

fig, ax = plt.subplots(figsize=(10, 6))
categories_fin = ['Receita Atual\n(2016-2018)', 'Com Recompra', '+ Cross-Sell', '+ Frete\n= PROJETADA']
cumulative = [
    total_receita_fin,
    total_receita_fin + receita_incremental_recompra,
    total_receita_fin + receita_incremental_recompra + receita_incremental_crosssell,
    receita_projetada_com,
]
cumulative_m = [v / 1_000_000 for v in cumulative]

bars = ax.bar(categories_fin, cumulative_m, color=[PALETTE[7], PALETTE[2], PALETTE[0], PALETTE[1]],
              edgecolor='white', width=0.6)
ax.set_ylabel('Receita (R$ milhoes)')
ax.set_title('Receita Projetada com Iniciativas Estrategicas Combinadas',
             fontsize=14, fontweight='bold')
for bar, val in zip(bars, cumulative_m):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'R$ {val:.1f}M', ha='center', fontsize=11, fontweight='bold')

ax.annotate(f'+{aumento_pct:.1f}%', xy=(3, cumulative_m[3]),
            xytext=(3, cumulative_m[3] + 0.8),
            fontsize=14, fontweight='bold', color=PALETTE[2],
            ha='center', va='bottom',
            arrowprops=dict(arrowstyle='->', color=PALETTE[2]))
fig.tight_layout()
plt.show()

---
## 12. Resumo e Conclusoes

### KPIs Consolidados

| Indicador | Valor |
|---|---|
| Receita Total | R$ 15,7 milhoes |
| Total de Pedidos | 98.200 |
| Ticket Medio | R$ 160,24 |
| Clientes Unicos | 93.358 |
| Mediana Lead Time | 10,2 dias |
| Taxa de Atraso | 8,1% |
| Score Medio | 4,16 / 5,0 |
| NPS Geral | 66,1 |
| Taxa de Recompra | 3,0% |

### Principais Insights

1. **Crescimento acelerado**: de centenas para 7 mil pedidos/mes, impulsionado por volume
2. **Logistica como driver de satisfacao**: atrasos derrubam o score de 4,3 para 2,5
3. **Recompra praticamente inexistente**: 97% dos clientes compram apenas uma vez
4. **Text mining confirma**: "entrega" e o tema #1 tanto em reviews negativas quanto positivas
5. **Oportunidade financeira**: iniciativas combinadas podem gerar +R$ 2M (+12,9%)

### Recomendacoes Estrategicas (priorizadas)

1. **Programa de fidelizacao** - CRM, incentivos para segunda compra (+R$ 448K)
2. **Cross-sell inteligente** - Motor de recomendacao baseado em padroes de compra (+R$ 1,4M)
3. **Otimizacao logistica regional** - Centros de distribuicao no Norte/Nordeste
4. **Gestao de sellers** - Capacitacao, premiacoes e descredenciamento
5. **Otimizacao de frete** - Negociacao em categorias com frete >30% do preco (+R$ 95K)

In [ ]:
print('='*60)
print('ANALISE CONCLUIDA')
print('='*60)
print(f'Total de graficos gerados: 27')
print(f'Trilhas analiticas cobertas: 9')
print(f'\nTech Challenge - Brazilian E-Commerce')
print(f'Base: Brazilian E-Commerce Public Dataset by Olist')